In [16]:
import os
import re
import pandas as pd
import plotly.express as px

In [17]:
datapath = r"/Volumes/TytellLab$/NewData/ElongateFoils/ElongateFoil_DorsalandAnalFins"
pivdatapath = os.path.join(datapath, "Processed data")

In [18]:
pivfiles = []
for root, dirs, files in os.walk(pivdatapath):
    for file in files:
        if file.endswith(".txt"):
            pivfiles.append(os.path.join(root, file))

print(f"Found {len(pivfiles)} PIV files.")

Found 160 PIV files.


In [19]:
pivfiles[50]

'/Volumes/TytellLab$/NewData/ElongateFoils/ElongateFoil_DorsalandAnalFins/Processed data/2026-06-22/flow15_0/flow15_0_0001_0010.txt'

In [ ]:
data = []
for fn1 in pivfiles:
    # We need to handle double numbers in the filename
    # flow3_0_0001.txt or flow3_0_0001_0002.txt
    # the last number is the frame number, and if there is another four digit number, it means
    # that we ran two PIV analyses on the same image, and the second one is the one we want to use.
    m = re.search(r"(\d{4})?_(\d{4})\.txt", fn1)
    if not m:
        print(f"Could not find frame number in filename: {fn1}")
        continue

    frame_number = int(m.groups()[1])
    if m.groups()[0] is None:
        rep = 1
    else:
        rep = 2

    data1 = pd.read_csv(fn1, header=2,
                        names=["x", "y", "u", "v", "vectype"])
    data1["filename"] = os.path.basename(fn1)
    data1["frame_number"] = frame_number
    data1["rep"] = rep
    
    m = re.search(r"flow(\d+_\d+)", fn1)
    if m:
        ms1 = re.sub("_", ".", m.groups()[0])
        data1["motor_speed"] = float(ms1)
    else:
        print(f"Could not find motor speed in filename: {fn1}")
        data1["motor_speed"] = None
    
    data.append(data1)

data = pd.concat(data, ignore_index=True)


Get only the last rep

In [20]:
data = data[
    data['rep'] == data.groupby(["motor_speed"])["rep"].transform("max")
    ]

In [21]:
data.head()

,x,y,u,v,vectype,filename,frame_number,rep,motor_speed
88,0.072221,0.029797,-0.180716,0.009292,1,flow11_0_0001_0001.txt,1,2,11.0
89,0.072221,0.037878,-0.183131,0.006452,1,flow11_0_0001_0001.txt,1,2,11.0
90,0.072221,0.045959,-0.172665,-0.000175,1,flow11_0_0001_0001.txt,1,2,11.0
91,0.072221,0.054040,-0.171867,0.004416,1,flow11_0_0001_0001.txt,1,2,11.0
92,0.072221,0.062120,-0.140679,-0.001927,1,flow11_0_0001_0001.txt,1,2,11.0


Only keep vectype == 1, which means it's a good vector

In [22]:
data = data[data["vectype"] == 1]

Average over x values

In [26]:
meandata = data.groupby(["motor_speed", "y"])[["x", "u", "v"]].median(skipna=True, numeric_only=True)

In [27]:
meandata.head()

x         u         v
motor_speed y                                     
3.0         0.029797  0.076261 -0.032409  0.000012
            0.037878  0.076261 -0.043457 -0.000607
            0.045959  0.076261 -0.040809  0.001137
            0.054040  0.076261 -0.032955  0.000287
            0.062120  0.076261 -0.040104  0.000773

Get rid of the nested index for the rows

In [28]:
meandata.reset_index(inplace=True)

In [29]:
meandata.head()

,motor_speed,y,x,u,v
0,3.0,0.029797,0.076261,-0.032409,0.000012
1,3.0,0.037878,0.076261,-0.043457,-0.000607
2,3.0,0.045959,0.076261,-0.040809,0.001137
3,3.0,0.054040,0.076261,-0.032955,0.000287
4,3.0,0.062120,0.076261,-0.040104,0.000773


In [30]:
fig = px.line(meandata, x="y", y="u", color=meandata["motor_speed"])
             # color_continuous_scale='Viridis') # doesn't work even though Claude said it would
fig.show()


Here's the overall mean of medians for each flow speed

In [34]:
framemed = data.groupby(["motor_speed", "frame_number"])[["u"]].median()
framemed.groupby("motor_speed")["u"].mean(skipna=True)

motor_speed
3.0    -0.040178
5.0    -0.016090
7.0    -0.013009
9.0    -0.137035
11.0   -0.185876
13.0   -0.205644
15.0   -0.256220
17.0   -0.275099
19.0   -0.343897
Name: u, dtype: float64